# 生命周期

Starlette 应用程序可以注册一个生命周期处理程序来处理在应用程序启动之前或应用程序关闭时需要运行的代码。

In [ ]:
import contextlib

from starlette.applications import Starlette


@contextlib.asynccontextmanager
async def lifespan(app):
    async with some_async_resource():
        print("Run at startup!")
        yield
        print("Run on shutdown!")


routes = [
    ...
]

app = Starlette(routes=routes, lifespan=lifespan)

在生命周期结束之前，`Starlette` 将不会开始处理任何传入请求。

一旦所有连接都已关闭并且任何正在进行的后台任务都已完成，生命周期拆卸就会运行。

考虑用于`anyio.create_task_group()` 管理异步任务。

## 寿命状态
生命周期有`state`的概念，它是一个字典，可以用来在生命周期和请求之间共享对象。


In [ ]:
import contextlib
from typing import AsyncIterator, TypedDict

import httpx
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import PlainTextResponse
from starlette.routing import Route


class State(TypedDict):
    http_client: httpx.AsyncClient


@contextlib.asynccontextmanager
async def lifespan(app: Starlette) -> AsyncIterator[State]:
    async with httpx.AsyncClient() as client:
        yield {"http_client": client}


async def homepage(request: Request) -> PlainTextResponse:
    client = request.state.http_client
    response = await client.get("https://www.example.com")
    return PlainTextResponse(response.text)


app = Starlette(
    lifespan=lifespan,
    routes=[Route("/", homepage)]
)

state请求中接收到的信息是生命周期处理程序中接收到的状态的浅表副本。

## 测试中的运行寿命
您应该将其用作TestClient上下文管理器，以确保调用生命周期。


In [ ]:
from example import app
from starlette.testclient import TestClient


def test_homepage():
    with TestClient(app) as client:
        # Application's lifespan is called on entering the block.
        response = client.get("/")
        assert response.status_code == 200

    # And the lifespan's teardown is run when exiting the block.